In [ ]:
import os
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics     import classification_report, roc_auc_score, f1_score, recall_score
from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils    import get_model_train_eval


In [5]:
# 데이터 로딩 및 기본 전처리
train, test = load_data()
X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=['ID'], axis=1)

In [6]:
# zero_count_rate 제거했을때 corr()에서 제거될 컬럼수 149개 잔존
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.99)

# var3 처리
X_features['var3'] = X_features['var3'].replace(-999999, 2)



Train Data Analysis (Threshold: 99.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

In [ ]:
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, recall_score

thresholds = [0.85, 0.90, 0.95]

print(f"zero_count_rate > 0.99 이상 제거 후 : \n")
results = []
drop_columns = {}

# 1. 상관계수 행렬 계산
corr_matrix = X_features.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# 2. HyperOpt으로 저장된 모델 객체 불러오기
with open("models/XGBoost_99per_hyperopt.pkl", "rb") as f:
    xgb_clf = pickle.load(f)   # 이미 학습된 XGBClassifier 객체

with open("models/LogisticRegression_99per_hyperopt.pkl", "rb") as f:
    log_clf = pickle.load(f)   # 이미 학습된 LogisticRegression 객체

for th in thresholds:        
    # 3. threshold 이상인 컬럼 drop
    to_drop = [column for column in upper.columns if any(upper[column] > th)]
    drop_columns[th] = to_drop
    X_reduced      = X_features.drop(columns=to_drop)
    X_test_reduced = X_test.drop(columns=to_drop)
    
    # 4. 학습/검증 데이터 분리 (먼저 split)
    X_train, X_val, y_train, y_val = data_split(X_reduced, y_labels)
    
    # 🔍 샘플 수 검증
    if X_train.shape[0] != y_train.shape[0]:
        raise ValueError(f"Train set mismatch: X_train={X_train.shape[0]}, y_train={y_train.shape[0]}")
    if X_val.shape[0] != y_val.shape[0]:
        raise ValueError(f"Validation set mismatch: X_val={X_val.shape[0]}, y_val={y_val.shape[0]}")
    
    # 5. 스케일링은 split 이후에 적용
    X_train_scaled, X_val_scaled, scaler = scale_data(X_train, X_val)
    
    # -------------------------------
    # XGBoost (저장된 모델 객체 사용, 필요 시 재학습)
    xgb_clf.fit(X_train, y_train)
    y_pred_xgb = xgb_clf.predict(X_val)
    y_proba_xgb = xgb_clf.predict_proba(X_val)[:,1]
    
    results.append({
        "threshold": th,
        "dropped_features": len(to_drop),
        "Model": "XGBoost_HyperOpt",
        "AUC": roc_auc_score(y_val, y_proba_xgb),
        "F1": f1_score(y_val, y_pred_xgb),
        "Recall": recall_score(y_val, y_pred_xgb)
    })
    
    # -------------------------------
    # Logistic Regression (저장된 모델 객체 사용, 스케일링된 데이터 필요)
    log_clf.fit(X_train_scaled, y_train)
    y_pred_log = log_clf.predict(X_val_scaled)
    y_proba_log = log_clf.predict_proba(X_val_scaled)[:,1]
    
    results.append({
        "threshold": th,
        "dropped_features": len(to_drop),
        "Model": "LogisticRegression_HyperOpt",
        "AUC": roc_auc_score(y_val, y_proba_log),
        "F1": f1_score(y_val, y_pred_log),
        "Recall": recall_score(y_val, y_pred_log)
    })

# 결과 DataFrame으로 정리
results_df = pd.DataFrame(results)
print(results_df)

'''
  * 핵심 수정사항
    - 스케일링 위치 변경: data_split() 이후에 scale_data(X_train, X_val) 적용
    - LogisticRegression은 X_train_scaled, X_val_scaled 사용
    - XGBoost는 스케일링 없이 원본 X_train, X_val 사용
    - 샘플 수 mismatch를 자동으로 검증하는 로직 추가
'''

zero_count_rate > 0.99 이상 제거 후 : 

   threshold  dropped_features                        Model       AUC  \
0       0.85                87             XGBoost_HyperOpt  0.851085   
1       0.85                87  LogisticRegression_HyperOpt  0.800873   
2       0.90                73             XGBoost_HyperOpt  0.852748   
3       0.90                73  LogisticRegression_HyperOpt  0.803065   
4       0.95                53             XGBoost_HyperOpt  0.852041   
5       0.95                53  LogisticRegression_HyperOpt  0.806344   

         F1    Recall  
0  0.009868  0.004983  
1  0.157968  0.749169  
2  0.009852  0.004983  
3  0.158485  0.750831  
4  0.006590  0.003322  
5  0.161711  0.772425  


In [ ]:
# 데이터셋 샘플 수 풀일치 문제 (shape 확인)

# print("X_train:", X_train.shape, "y_train:", y_train.shape)
# print("X_val:", X_val.shape, "y_val:", y_val.shape)
# print("X_train_scaled:", X_train_scaled.shape, "X_test_scaled:", X_test_scaled.shape)

X_train: (60816, 62) y_train: (60816,)
X_val: (15204, 62) y_val: (15204,)
X_train_scaled: (76020, 62) X_test_scaled: (75818, 62)
